In [10]:
import json
import os
import shutil
import site
import subprocess
import sys
import tempfile
import textwrap
import uuid
from contextlib import contextmanager
from pathlib import Path
from types import SimpleNamespace

import h5py
import minari
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display



In [11]:
import shutil
import site

PROJECT_ROOT = Path.cwd()
MINARI_DATASETS_PATH = PROJECT_ROOT / ".minari" / "datasets"
D4RL_PYTHON = Path(r"C:\Users\diank\venv_d4rl\Scripts\python.exe")
MUJOCO_PATH = Path(r"C:\Users\diank\.mujoco\mujoco210")
D4RL_SOURCE = PROJECT_ROOT / "d4rl"
LOCAL_MUJOCO_PY_ROOT = PROJECT_ROOT / ".local_mujoco_py"
DATASET_ID = "analysis/maze2d/medium-raw-v0"
MAZE_SPEC_NAME = "MEDIUM_MAZE_EVAL"  # Numerically validated against maze2d-medium.hdf5
EPISODE_ID = 0
STRIDE = 6

In [12]:
MAZE_SPECS = {
    "OPEN": r"#######\#OOOOO#\#OOGOO#\#OOOOO#\#######",
    "U_MAZE": r"#####\#GOO#\###O#\#OOO#\#####",
    "U_MAZE_EVAL": r"#####\#OOG#\#O###\#OOO#\#####",
    "SMALL_MAZE": r"######\#OOOO#\#O##O#\#OOOO#\######",
    "MEDIUM_MAZE": r"########\#OO##OO#\#OO#OOO#\##OOO###\#OO#OOO#\#O#OO#O#\#OOO#OG#\########",
    "MEDIUM_MAZE_EVAL": r"########\#OOOOOG#\#O#O##O#\#OOOO#O#\###OO###\#OOOOOO#\#OO##OO#\########",
    "LARGE_MAZE": r"############\#OOOO#OOOOO#\#O##O#O#O#O#\#OOOOOO#OOO#\#O####O###O#\#OO#O#OOOOO#\##O#O#O#O###\#OO#OOO#OGO#\############",
    "LARGE_MAZE_EVAL": r"############\#OO#OOO#OGO#\##O###O#O#O#\#OO#O#OOOOO#\#O##O#OO##O#\#OOOOOO#OOO#\#O##O#O#O###\#OOOO#OOOOO#\############",
}

QPOS_WORLD_OFFSET = np.array([1.2, 1.2], dtype=np.float32)
QPOS_WALL_CENTER_OFFSET = np.array([-0.2, -0.2], dtype=np.float32)
WALL_HALF_EXTENT = 0.5
PARTICLE_RADIUS = 0.1

In [13]:
os.environ["MINARI_DATASETS_PATH"] = str(MINARI_DATASETS_PATH)
dataset = minari.load_dataset(DATASET_ID)
episode = dataset[EPISODE_ID]
episode

EpisodeData(id=0, total_steps=124, observations=ndarray of shape (124, 4) and dtype float32, actions=ndarray of shape (124, 2) and dtype float32, rewards=ndarray of 124 floats, terminations=ndarray of 124 bools, truncations=ndarray of 124 bools, infos=dict with the following keys: ['goal', 'qpos', 'qvel', 'raw_index'])

In [14]:

positions = np.asarray(episode.infos["qpos"], dtype=np.float32)
episode_actions = np.asarray(episode.actions, dtype=np.float32)
goal = np.asarray(episode.infos["goal"][0], dtype=np.float32)

print("Loaded dataset:", DATASET_ID)
print("Episode length:", len(episode_actions))

Loaded dataset: analysis/maze2d/medium-raw-v0
Episode length: 124


In [15]:
episode_summary = {
    "observations_shape": episode.observations.shape,
    "actions_shape": episode.actions.shape,
    "goal_shape": episode.infos["goal"].shape,
    "qpos_shape": episode.infos["qpos"].shape,
    "qvel_shape": episode.infos["qvel"].shape,
    "info_keys": list(episode.infos.keys()),
    'rewards': episode.rewards.shape,
}
episode_summary

{'observations_shape': (124, 4),
 'actions_shape': (124, 2),
 'goal_shape': (124, 2),
 'qpos_shape': (124, 2),
 'qvel_shape': (124, 2),
 'info_keys': ['goal', 'qpos', 'qvel', 'raw_index'],
 'rewards': (124,)}

In [16]:
episode_table = pd.DataFrame(
    {
        "t": np.arange(len(episode.actions), dtype=np.int32),
        "obs_x": episode.observations[:, 0],
        "obs_y": episode.observations[:, 1],
        "obs_vx": episode.observations[:, 2],
        "obs_vy": episode.observations[:, 3],
        "qpos_x": episode.infos["qpos"][:, 0],
        "qpos_y": episode.infos["qpos"][:, 1],
        "goal_x": episode.infos["goal"][:, 0],
        "goal_y": episode.infos["goal"][:, 1],
        "action_x": episode.actions[:, 0],
        "action_y": episode.actions[:, 1],
        "rewards": episode.rewards,
        "terminated": episode.terminations,
        "truncated": episode.truncations,
    }
)
episode_table.head(10)

,t,obs_x,obs_y,obs_vx,obs_vy,qpos_x,qpos_y,goal_x,goal_y,action_x,action_y,rewards,terminated,truncated
0,0,4.944978,4.084466,0.008865,0.045366,4.944978,4.084466,5.995181,0.912286,0.125213,-0.681920,0.112269,False,False
1,1,4.945365,4.083295,0.038665,-0.117150,4.945365,4.083295,5.995181,0.912286,-1.000000,-1.000000,0.112175,False,False
2,2,4.943369,4.079744,-0.199591,-0.355035,4.943369,4.079744,5.995181,0.912286,-0.244802,-0.592145,0.111719,False,False
3,3,4.940795,4.074792,-0.257418,-0.495217,4.940795,4.074792,5.995181,0.912286,-1.000000,-1.000000,0.111097,False,False
4,4,4.935845,4.067470,-0.494969,-0.732202,4.935845,4.067470,5.995181,0.912286,-0.401482,-0.988491,0.110124,False,False
5,5,4.929951,4.057811,-0.589409,-0.965881,4.929951,4.057811,5.995181,0.912286,-0.059175,-1.000000,0.108886,False,False
6,6,4.923930,4.045794,-0.602098,-1.201744,4.923930,4.045794,5.995181,0.912286,0.397371,-0.789803,0.107433,False,False
7,7,4.918870,4.031924,-0.506025,-1.386984,4.918870,4.031924,5.995181,0.912286,0.078696,-1.000000,0.105877,False,False
8,8,4.914009,4.015706,-0.486077,-1.621845,4.914009,4.015706,5.995181,0.912286,-0.315761,-0.831028,0.104138,False,False
9,9,4.908408,3.997547,-0.560122,-1.815903,4.908408,3.997547,5.995181,0.912286,0.818448,-1.000000,0.102217,False,False


In [17]:
def parse_maze_rows(maze_str):
    return maze_str.strip().split("\\")

def parse_maze_logic(maze_str):
    maze_rows = parse_maze_rows(maze_str)
    return np.array([[1 if char == "#" else 0 for char in row_text] for row_text in maze_rows], dtype=np.int8)

maze_spec_name = MAZE_SPEC_NAME
def qpos_free_cell_reference_xy(maze_logic):
    return np.argwhere(maze_logic == 0).astype(np.float32)
maze_logic = parse_maze_logic(MAZE_SPECS[maze_spec_name])
free_cell_reference_xy = qpos_free_cell_reference_xy(maze_logic)
maze_logic = parse_maze_logic(MAZE_SPECS[maze_spec_name])

In [18]:
DISCRETE_GRID_SHAPE = tuple(int(v) for v in maze_logic.shape)
def make_discrete_maze(maze_logic, grid_shape=None):
    if grid_shape is None:
        grid_shape = tuple(int(v) for v in maze_logic.shape)
    if tuple(grid_shape) != tuple(int(v) for v in maze_logic.shape):
        raise ValueError("This notebook keeps the discrete maze in the same logical (maze_x, maze_y) coordinates as qpos.")
    return np.asarray(maze_logic, dtype=np.int32).copy()


def discretize_points(points_xy, free_cell_reference_xy):
    points_xy = np.asarray(points_xy, dtype=np.float32)
    distances = ((points_xy[:, None, :] - free_cell_reference_xy[None, :, :]) ** 2).sum(axis=2)
    nearest_indices = distances.argmin(axis=1)
    return free_cell_reference_xy[nearest_indices].astype(np.int16)


discrete_maze = make_discrete_maze(maze_logic, grid_shape=DISCRETE_GRID_SHAPE)


In [23]:
from collections import deque

DISCRETE_PATH_DATASET_PATH = PROJECT_ROOT / "maze2d_discrete_trajectory_paths_8x8_v2.npz"


def collapse_repeated_cells(path):
    path = np.asarray(path, dtype=np.int16)
    if len(path) == 0:
        return path.reshape(0, 2)
    keep = np.ones(len(path), dtype=bool)
    keep[1:] = np.any(path[1:] != path[:-1], axis=1)
    return path[keep]

def discretize_points(points_xy, free_cell_reference_xy):
    points_xy = np.asarray(points_xy, dtype=np.float32)
    distances = ((points_xy[:, None, :] - free_cell_reference_xy[None, :, :]) ** 2).sum(axis=2)
    nearest_indices = distances.argmin(axis=1)
    return free_cell_reference_xy[nearest_indices].astype(np.int16)
def shortest_grid_path(maze, start_cell, end_cell):
    start_cell = tuple(np.asarray(start_cell, dtype=np.int16).tolist())
    end_cell = tuple(np.asarray(end_cell, dtype=np.int16).tolist())

    if start_cell == end_cell:
        return np.asarray([start_cell], dtype=np.int16)

    grid_x, grid_y = maze.shape
    queue = deque([start_cell])
    previous = {start_cell: None}

    while queue:
        cell_x, cell_y = queue.popleft()
        if (cell_x, cell_y) == end_cell:
            break

        for next_x, next_y in ((cell_x - 1, cell_y), (cell_x + 1, cell_y), (cell_x, cell_y - 1), (cell_x, cell_y + 1)):
            if not (0 <= next_x < grid_x and 0 <= next_y < grid_y):
                continue
            if maze[next_x, next_y] != 0:
                continue
            if (next_x, next_y) in previous:
                continue
            previous[(next_x, next_y)] = (cell_x, cell_y)
            queue.append((next_x, next_y))

    if end_cell not in previous:
        raise RuntimeError(f"No free-cell path found between {start_cell} and {end_cell}.")

    path = []
    current = end_cell
    while current is not None:
        path.append(current)
        current = previous[current]
    path.reverse()
    return np.asarray(path, dtype=np.int16)

def connect_discrete_path(path, maze):
    collapsed = collapse_repeated_cells(path)
    if len(collapsed) == 0:
        return collapsed

    connected_segments = [collapsed[:1]]
    for next_cell in collapsed[1:]:
        segment = shortest_grid_path(maze, connected_segments[-1][-1], next_cell)
        connected_segments.append(segment[1:])

    connected_path = np.concatenate(connected_segments, axis=0)
    return collapse_repeated_cells(connected_path)

def build_discrete_path_dataset(minari_dataset, free_cell_reference_xy, grid_shape, maze, maze_name):
    episode_ids = np.arange(len(minari_dataset), dtype=np.int32)
    paths = np.empty(len(minari_dataset), dtype=object)
    raw_lengths = np.zeros(len(minari_dataset), dtype=np.int32)
    path_lengths = np.zeros(len(minari_dataset), dtype=np.int32)
    start_cells = np.zeros((len(minari_dataset), 2), dtype=np.int16)
    end_cells = np.zeros((len(minari_dataset), 2), dtype=np.int16)
    goal_cells = np.zeros((len(minari_dataset), 2), dtype=np.int16)

    for episode_id in episode_ids:
        episode = minari_dataset[int(episode_id)]
        raw_path = discretize_points(episode.infos["qpos"], free_cell_reference_xy)
        connected_path = connect_discrete_path(raw_path, maze)
        discrete_goal_cell = discretize_points(episode.infos["goal"][:1], free_cell_reference_xy)[0]

        paths[int(episode_id)] = connected_path.astype(np.int16)
        raw_lengths[int(episode_id)] = len(raw_path)
        path_lengths[int(episode_id)] = len(connected_path)
        start_cells[int(episode_id)] = connected_path[0]
        end_cells[int(episode_id)] = connected_path[-1]
        goal_cells[int(episode_id)] = discrete_goal_cell.astype(np.int16)

    return {
        "maze_name": np.array([maze_name]),
        "coordinate_convention": np.array(["logical_qpos_xy"]),
        "grid_shape": np.asarray(grid_shape, dtype=np.int32),
        "maze": np.asarray(maze, dtype=np.int8),
        "episode_ids": episode_ids,
        "raw_lengths": raw_lengths,
        "path_lengths": path_lengths,
        "start_cells": start_cells,
        "end_cells": end_cells,
        "goal_cells": goal_cells,
        "paths": paths,
    }

discrete_path_dataset = build_discrete_path_dataset(
    minari_dataset=dataset,
    free_cell_reference_xy=free_cell_reference_xy,
    grid_shape=DISCRETE_GRID_SHAPE,
    maze=discrete_maze,
    maze_name=maze_spec_name,
)


np.savez(DISCRETE_PATH_DATASET_PATH, **discrete_path_dataset)


discrete_path_dataset



{'maze_name': array(['MEDIUM_MAZE_EVAL'], dtype='<U16'),
 'coordinate_convention': array(['logical_qpos_xy'], dtype='<U15'),
 'grid_shape': array([8, 8], dtype=int32),
 'maze': array([[1, 1, 1, 1, 1, 1, 1, 1],
        [1, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 1, 0, 1, 1, 0, 1],
        [1, 0, 0, 0, 0, 1, 0, 1],
        [1, 1, 1, 0, 0, 1, 1, 1],
        [1, 0, 0, 0, 0, 0, 0, 1],
        [1, 0, 0, 1, 1, 0, 0, 1],
        [1, 1, 1, 1, 1, 1, 1, 1]], dtype=int8),
 'episode_ids': array([    0,     1,     2, ..., 12557, 12558, 12559],
       shape=(12560,), dtype=int32),
 'raw_lengths': array([124, 275,  36, ..., 179, 391,   1], shape=(12560,), dtype=int32),
 'path_lengths': array([ 5, 10,  2, ...,  7, 13,  1], shape=(12560,), dtype=int32),
 'start_cells': array([[5, 4],
        [6, 1],
        [1, 1],
        ...,
        [1, 2],
        [3, 6],
        [5, 6]], shape=(12560, 2), dtype=int16),
 'end_cells': array([[6, 1],
        [1, 1],
        [1, 2],
        ...,
        [3, 6],
        [5